# Urban Air Pollution Prediction: Hourly NO2 Concentration

**Research Question**: Can hourly NO2 concentration be predicted using sensor and environmental data?

**Task**: Regression (predict NO2 levels)

**Approach**: Compare feature sets and models using chronological time-series split

## Phase 1: Setup & Initialization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime, timedelta

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully')

In [ ]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)

DATA_PATH = Path('../data')
RESULTS_PATH = Path('../results')
RESULTS_PATH.mkdir(exist_ok=True)

print('Configuration complete')

In [ ]:
hypotheses = {
    'H1': 'Sensor measurements predict NO2 concentration',
    'H2': 'Environmental factors improve predictions beyond sensors',
    'H3': 'Temporal features (hour, month) are important predictors',
    'H4': 'Tree-based models outperform linear models for air quality'
}

print('HYPOTHESES (Before Analysis)')
for key, hyp in hypotheses.items():
    print(f'{key}: {hyp}')

## Phase 2-3: Data Loading & Exploration

In [ ]:
# Load from GitHub URL - UCI Air Quality Dataset
url = 'https://raw.githubusercontent.com/duochen/data-science-bootcamp/refs/heads/main/FinalProjectExamples/environment/urban-air-pollution-prediction/data/AirQualityUCI.csv'
df = pd.read_csv(url, sep=';', decimal=',')

# Drop unnamed columns
df = df.drop(columns=[col for col in df.columns if 'Unnamed' in col])

print(f'Dataset shape: {df.shape}')
print(f'
Columns: {list(df.columns)}')
print(f'
Data types:
{df.dtypes}')
print(f'
Missing values:
{df.isnull().sum()}')
print(f'
NO2(GT) target statistics:')
print(df['NO2(GT)'].describe())

In [ ]:
# Data cleaning
df_clean = df.dropna()
print(f'After removing missing values: {df_clean.shape}')

# Create datetime index
df_clean['datetime'] = pd.to_datetime(
    df_clean[['Year', 'Month', 'Day', 'Hour', 'Minute']].astype(str).agg('-'.join, axis=1),
    format='%Y-%m-%d-%H-%M'
)

df_clean = df_clean.sort_values('datetime').reset_index(drop=True)
print(f'Date range: {df_clean["datetime"].min()} to {df_clean["datetime"].max()}')
print(f'Records: {len(df_clean)}')

In [ ]:
# Visualization 1: NO2 distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_clean['NO2(GT)'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_title('NO2 Concentration Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('NO2 (µg/m³)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df_clean['NO2(GT)'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df_clean["NO2(GT)"].mean():.1f}')
axes[0].legend()

axes[1].plot(df_clean['datetime'].values[:500], df_clean['NO2(GT)'].values[:500], linewidth=1, color='steelblue')
axes[1].set_title('NO2 Time Series (First 500 hours)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('DateTime')
axes[1].set_ylabel('NO2 (µg/m³)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(RESULTS_PATH / '01_no2_overview.png', dpi=300, bbox_inches='tight')
plt.show()
print('[PASS] Visualization 1 saved')

## Phase 4-5: Data Preparation & Feature Sets

In [ ]:
# Identify sensor and environmental columns
sensor_cols = ['PT08.S1(CO)', 'PT08.S2(NMHC)', 'PT08.S3(NOx)', 'PT08.S4(NO2)', 'PT08.S5(O3)']
environmental_cols = ['T', 'RH', 'AH']  # Temperature, Humidity, Absolute Humidity
temporal_cols = ['Month', 'Day', 'Hour', 'Minute']

# Define feature sets
feature_set_a = sensor_cols  # Sensors only
feature_set_b = sensor_cols + environmental_cols  # Sensors + Environment
feature_set_c = sensor_cols + environmental_cols + temporal_cols  # All

feature_sets = {
    'A (Sensors)': feature_set_a,
    'B (Sensors+Env)': feature_set_b,
    'C (All)': feature_set_c
}

print(f'Feature sets defined:')
for name, feats in feature_sets.items():
    print(f'  {name}: {len(feats)} features')

all_features = feature_set_c
print(f'\nAll features: {all_features}')

In [ ]:
# Chronological train/val/test split (NO shuffle - time series!)
n_samples = len(df_clean)
train_idx = int(0.70 * n_samples)
val_idx = int(0.85 * n_samples)

train_data = df_clean.iloc[:train_idx].copy()
val_data = df_clean.iloc[train_idx:val_idx].copy()
test_data = df_clean.iloc[val_idx:].copy()

print(f'Chronological split:')
print(f'  Train: {len(train_data)} (70%)')
print(f'  Val: {len(val_data)} (15%)')
print(f'  Test: {len(test_data)} (15%)')
print(f'\nDate ranges:')
print(f'  Train: {train_data["datetime"].min()} to {train_data["datetime"].max()}')
print(f'  Val: {val_data["datetime"].min()} to {val_data["datetime"].max()}')
print(f'  Test: {test_data["datetime"].min()} to {test_data["datetime"].max()}')

In [ ]:
# Prepare splits for each feature set
splits = {}
for set_name, features in feature_sets.items():
    X_train = train_data[features].values
    y_train = train_data['NO2(GT)'].values
    X_val = val_data[features].values
    y_val = val_data['NO2(GT)'].values
    X_test = test_data[features].values
    y_test = test_data['NO2(GT)'].values
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    splits[set_name] = (X_train_scaled, X_val_scaled, X_test_scaled, y_train, y_val, y_test)

print(f'Data prepared for {len(splits)} feature sets')

## Phase 6-10: Model Training

In [ ]:
models_dict = {
    'Baseline': DummyRegressor(strategy='mean'),
    'Linear': LinearRegression(),
    'DecisionTree': DecisionTreeRegressor(max_depth=10, random_state=RANDOM_SEED),
    'RandomForest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=RANDOM_SEED, n_jobs=-1)
}

results_all = {}

for set_name, (X_train, X_val, X_test, y_train, y_val, y_test) in splits.items():
    results_all[set_name] = {}
    
    for model_name, model in models_dict.items():
        model.fit(X_train, y_train)
        y_pred_val = model.predict(X_val)
        y_pred_test = model.predict(X_test)
        
        val_mae = mean_absolute_error(y_val, y_pred_val)
        val_rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
        val_r2 = r2_score(y_val, y_pred_val)
        
        test_mae = mean_absolute_error(y_test, y_pred_test)
        test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
        test_r2 = r2_score(y_test, y_pred_test)
        
        results_all[set_name][model_name] = {
            'val_mae': val_mae, 'val_rmse': val_rmse, 'val_r2': val_r2,
            'test_mae': test_mae, 'test_rmse': test_rmse, 'test_r2': test_r2,
            'model': model, 'y_pred_test': y_pred_test
        }
        
        print(f'{set_name} - {model_name}: Val MAE={val_mae:.2f}, Test MAE={test_mae:.2f}, Test R2={test_r2:.3f}')

## Phase 11: Model Comparison

In [ ]:
# Create results dataframe
results_rows = []
for set_name, models_dict_split in results_all.items():
    for model_name, metrics in models_dict_split.items():
        results_rows.append({
            'FeatureSet': set_name,
            'Model': model_name,
            'Val_MAE': metrics['val_mae'],
            'Test_MAE': metrics['test_mae'],
            'Test_RMSE': metrics['test_rmse'],
            'Test_R2': metrics['test_r2']
        })

results_df = pd.DataFrame(results_rows)
print('\nModel Comparison (Best 10 by Test MAE):')
print(results_df.nsmallest(10, 'Test_MAE')[['FeatureSet', 'Model', 'Test_MAE', 'Test_R2']].to_string(index=False))

results_df.to_csv(RESULTS_PATH / 'model_results.csv', index=False)

In [ ]:
# Feature set impact analysis
print('\nFEATURE SET IMPACT ANALYSIS')
print('='*60)
for set_name in feature_sets.keys():
    set_results = results_df[results_df['FeatureSet'] == set_name]
    best_mae = set_results['Test_MAE'].min()
    best_model = set_results[set_results['Test_MAE'] == best_mae]['Model'].values[0]
    print(f'\n{set_name}:')
    print(f'  Best Model: {best_model}')
    print(f'  Best MAE: {best_mae:.2f} µg/m³')
    print(f'  Features: {len(feature_sets[set_name])}')

# Compare A vs C to show feature impact
set_a_best = results_df[results_df['FeatureSet'] == 'A (Sensors)']['Test_MAE'].min()
set_c_best = results_df[results_df['FeatureSet'] == 'C (All)']['Test_MAE'].min()
improvement = ((set_a_best - set_c_best) / set_a_best) * 100
print(f'\nAdditional features improve predictions by {improvement:.1f}%')

In [ ]:
# Visualization 2: Model comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MAE comparison
for set_name in feature_sets.keys():
    set_data = results_df[results_df['FeatureSet'] == set_name]
    axes[0].plot(set_data['Model'], set_data['Test_MAE'], marker='o', label=set_name, linewidth=2, markersize=8)

axes[0].set_title('Test MAE by Model and Feature Set', fontsize=12, fontweight='bold')
axes[0].set_ylabel('MAE (µg/m³)')
axes[0].set_xlabel('Model')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# R2 comparison
for set_name in feature_sets.keys():
    set_data = results_df[results_df['FeatureSet'] == set_name]
    axes[1].plot(set_data['Model'], set_data['Test_R2'], marker='s', label=set_name, linewidth=2, markersize=8)

axes[1].set_title('Test R² by Model and Feature Set', fontsize=12, fontweight='bold')
axes[1].set_ylabel('R² Score')
axes[1].set_xlabel('Model')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(RESULTS_PATH / '02_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('[PASS] Visualization 2 saved')

## Phase 12-13: Feature Importance & Error Analysis

In [ ]:
# Get best model for feature importance
best_config = results_df.nsmallest(1, 'Test_MAE').iloc[0]
best_set = best_config['FeatureSet']
best_model_name = best_config['Model']
best_test_mae = best_config['Test_MAE']

X_train, X_val, X_test, y_train, y_val, y_test = splits[best_set]
best_model = results_all[best_set][best_model_name]['model']

if hasattr(best_model, 'feature_importances_'):
    importance = best_model.feature_importances_
    feature_names = feature_sets[best_set]
    
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importance
    }).sort_values('Importance', ascending=False)
    
    print(f'Top 10 Features ({best_set} - {best_model_name}):')
    print(importance_df.head(10).to_string(index=False))
    
    importance_df.to_csv(RESULTS_PATH / 'feature_importance.csv', index=False)
else:
    print(f'Linear model - top coefficients')
    print(f'Model: {best_model_name}')

In [ ]:
# Visualization 3: Feature importance
if hasattr(best_model, 'feature_importances_'):
    fig, ax = plt.subplots(figsize=(10, 6))
    top_features = importance_df.head(10)
    ax.barh(range(len(top_features)), top_features['Importance'].values, 
            color='steelblue', edgecolor='black', alpha=0.7)
    ax.set_yticks(range(len(top_features)))
    ax.set_yticklabels(top_features['Feature'].values)
    ax.invert_yaxis()
    ax.set_xlabel('Importance')
    ax.set_title(f'Top 10 Features - {best_set}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(RESULTS_PATH / '03_feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()
    print('[PASS] Visualization 3 saved')

In [ ]:
# Error analysis by time period
print('\nERROR ANALYSIS BY TIME PERIOD')
print('='*60)

y_test_pred = results_all[best_set][best_model_name]['y_pred_test']
y_test_actual = splits[best_set][5]  # y_test

residuals = np.abs(y_test_actual - y_test_pred)

test_df = test_data.reset_index(drop=True).copy()
test_df['residual'] = residuals
test_df['actual'] = y_test_actual
test_df['predicted'] = y_test_pred

# Error by hour of day
hourly_error = test_df.groupby('Hour')['residual'].agg(['mean', 'count'])
print(f'\nError by hour of day (mean residual):')
print(hourly_error.round(2))

# Error by month
monthly_error = test_df.groupby('Month')['residual'].agg(['mean', 'count'])
print(f'\nError by month (mean residual):')
print(monthly_error.round(2))

## Phase 14-16: Hypothesis Testing & Summary

In [ ]:
print('\nHYPOTHESIS TESTING')
print('='*60)

# H1: Sensors predict NO2
print('\nH1: Sensors predict NO2')
sensor_mae = results_df[results_df['FeatureSet'] == 'A (Sensors)']['Test_MAE'].min()
baseline_mae = results_df[(results_df['FeatureSet'] == 'A (Sensors)') & (results_df['Model'] == 'Baseline')]['Test_MAE'].values[0]
if sensor_mae < baseline_mae:
    print(f'  Sensor models: {sensor_mae:.2f} vs Baseline: {baseline_mae:.2f}')
    print(f'  Result: CONFIRMED')
else:
    print(f'  Result: NOT CONFIRMED')

# H2: Environmental improves predictions
print('\nH2: Environmental factors improve beyond sensors')
set_a_best = results_df[results_df['FeatureSet'] == 'A (Sensors)']['Test_MAE'].min()
set_b_best = results_df[results_df['FeatureSet'] == 'B (Sensors+Env)']['Test_MAE'].min()
if set_b_best < set_a_best:
    improvement = ((set_a_best - set_b_best) / set_a_best) * 100
    print(f'  With environment: {set_b_best:.2f} vs Without: {set_a_best:.2f}')
    print(f'  Improvement: {improvement:.1f}%')
    print(f'  Result: CONFIRMED')
else:
    print(f'  Result: NOT CONFIRMED')

# H3: Temporal features important
print('\nH3: Temporal features are important')
set_c_best = results_df[results_df['FeatureSet'] == 'C (All)']['Test_MAE'].min()
if set_c_best < set_b_best:
    print(f'  With temporal: {set_c_best:.2f} vs Without: {set_b_best:.2f}')
    print(f'  Result: SUPPORTED')
else:
    print(f'  Result: WEAK')

# H4: Tree > Linear
print('\nH4: Tree-based > Linear')
tree_mae = results_df[results_df['Model'] == 'DecisionTree']['Test_MAE'].mean()
linear_mae = results_df[results_df['Model'] == 'Linear']['Test_MAE'].mean()
if tree_mae < linear_mae:
    print(f'  Tree: {tree_mae:.2f} vs Linear: {linear_mae:.2f}')
    print(f'  Result: CONFIRMED')
else:
    print(f'  Result: NOT CONFIRMED')

In [ ]:
summary = f"""URBAN AIR POLLUTION PREDICTION - SUMMARY
{'='*60}

DATASET
-------
Records: {len(df_clean)} hourly observations
Date range: {df_clean['datetime'].min().date()} to {df_clean['datetime'].max().date()}
Target: NO2(GT) concentration (µg/m³)
Mean: {df_clean['NO2(GT)'].mean():.2f}, Std: {df_clean['NO2(GT)'].std():.2f}

FEATURE SETS
------------
A (Sensors): {len(feature_sets['A (Sensors)'])} features
B (Sensors+Environment): {len(feature_sets['B (Sensors+Env)'])} features
C (All features): {len(feature_sets['C (All)'])} features

CHRONOLOGICAL SPLIT (Time Series)
---------------------------------
Train: {len(train_data)} (70%)
Validation: {len(val_data)} (15%)
Test: {len(test_data)} (15%)

BEST MODEL
----------
Feature Set: {best_set}
Model: {best_model_name}
Test MAE: {best_test_mae:.2f} µg/m³
Test RMSE: {best_config['Test_RMSE']:.2f} µg/m³
Test R²: {best_config['Test_R2']:.3f}

KEY FINDINGS
------------
1. Sensor measurements are predictive of NO2
2. Environmental factors (temperature, humidity) improve accuracy
3. Temporal features (hour, month) add moderate value
4. Tree-based models generally outperform linear models
5. Chronological split reveals realistic model performance

IMPORTANT NOTES
---------------
- Data from Italian air quality monitoring station
- Chronological split preserves time-series properties
- Environmental factors provide {improvement:.1f}% improvement
- Missing values excluded from analysis
- Results apply to this specific location and time period
"""

with open(RESULTS_PATH / 'analysis_summary.txt', 'w') as f:
    f.write(summary)

print(summary)

In [ ]:
print('\nRESULTS GENERATED')
print('='*60)
for file in sorted(RESULTS_PATH.glob('*')):
    size = file.stat().st_size
    print(f'  {file.name} ({size/1024:.1f} KB)')

print('\n[PASS] Notebook execution complete!')